In [ ]:
# Connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#Load packages
import pandas as pd
import numpy as np

# I. Comparing Event Run Values

In [ ]:
# Using a function similar to what was used in the week’s exercise notebook.
def Run_Expectancy(path):

    RE = pd.read_csv(path)
    RE.drop(['Unnamed: 0'], axis=1, inplace=True)
    RE = RE[['home_team','away_team','half','gameId','batterName','batterId','event', 'start1B', 'start2B', 'start3B',\
             'end1B', 'end2B', 'end3B', 'startOuts','endOuts','runsFuture','runsOnPlay','outsInInning','venueId','batterPos']]
    RE['Start1'] = np.where(pd.isnull(RE['start1B']),0,1)
    RE['Start2'] = np.where(pd.isnull(RE['start2B']),0,1)
    RE['Start3'] = np.where(pd.isnull(RE['start3B']),0,1)
    RE['Start_State'] = (RE['Start1'].astype(str) + RE['Start2'].astype(str) + RE['Start3'].astype(str)+\
                          " " + RE['startOuts'].astype(str))
    RE['End1'] = np.where(pd.isnull(RE['end1B']),0,1)
    RE['End2'] = np.where(pd.isnull(RE['end2B']),0,1)
    RE['End3'] = np.where(pd.isnull(RE['end3B']),0,1)
    RE['End_State'] = (RE['End1'].astype(str) + RE['End2'].astype(str) + RE['End3'].astype(str) + \
                        " " + RE['endOuts'].astype(str))
    RE = RE[((RE.Start_State != RE.End_State) | (RE.runsOnPlay > 0)) & (RE.outsInInning == 3)]
    Start_RunExp = RE.groupby(['Start_State'])['runsFuture'].mean().reset_index().rename(columns={'runsFuture':'Start_RE'})
    RE = pd.merge(RE, Start_RunExp, on=['Start_State'], how='left')
    Base_State_3 = [pd.Series(['000 3', 0], index=Start_RunExp.columns),
                pd.Series(['001 3', 0], index=Start_RunExp.columns),
                pd.Series(['010 3', 0], index=Start_RunExp.columns),
                pd.Series(['011 3', 0], index=Start_RunExp.columns),
                pd.Series(['100 3', 0], index=Start_RunExp.columns),
                pd.Series(['101 3', 0], index=Start_RunExp.columns),
                pd.Series(['110 3', 0], index=Start_RunExp.columns),
                pd.Series(['111 3', 0], index=Start_RunExp.columns)]
    Start_RunExp = pd.concat([Start_RunExp, pd.DataFrame(Base_State_3)], ignore_index=True)
    End_RunExp  = Start_RunExp.rename(columns={'Start_State':'End_State', 'Start_RE':'End_RE'})
    RE = pd.merge(RE, End_RunExp, on=['End_State'], how='left')
    RE['Run_Value'] = RE['runsOnPlay'] + RE['End_RE'] - RE['Start_RE']

    return RE;

In [ ]:
# Read in MLBAM Data for 2014-2017 and calculate the run value for every event in 2014, 2015, 2016 and 2017

MLBAM14 = Run_Expectancy("/content/drive/MyDrive/Sports Performance Analytics Michigan/Course 2 - Moneyball and Beyond/Week 5 Content/Week 5 Assignment/MLBAM14.csv")
MLBAM15 = Run_Expectancy("/content/drive/MyDrive/Sports Performance Analytics Michigan/Course 2 - Moneyball and Beyond/Week 5 Content/Week 5 Assignment/MLBAM15 (1).csv")
MLBAM16 = Run_Expectancy("/content/drive/MyDrive/Sports Performance Analytics Michigan/Course 2 - Moneyball and Beyond/Week 5 Content/Week 5 Assignment/MLBAM16 (2).csv")
MLBAM17 = Run_Expectancy("/content/drive/MyDrive/Sports Performance Analytics Michigan/Course 2 - Moneyball and Beyond/Week 5 Content/Week 5 Assignment/MLBAM17 (2).csv")

In [ ]:
# Calculate the average run value for each type of event for every season
Event_Value14= MLBAM14.groupby(['event'])['Run_Value'].mean().reset_index().rename(columns = {"Run_Value": 'RV14'})
Event_Value15= MLBAM15.groupby(['event'])['Run_Value'].mean().reset_index().rename(columns = {"Run_Value": 'RV15'})
Event_Value16= MLBAM16.groupby(['event'])['Run_Value'].mean().reset_index().rename(columns = {"Run_Value": 'RV16'})
Event_Value17= MLBAM17.groupby(['event'])['Run_Value'].mean().reset_index().rename(columns = {"Run_Value": 'RV17'})

In [ ]:
# Merge the event level run values for each season into one data frame.
# The data frame should include the event name and then four columns with the run values (one for each season).
EV1 = pd.merge(Event_Value14,Event_Value15, on = ['event'])
EV2 = pd.merge(Event_Value16,Event_Value17, on = ['event'])
Event_Value= pd.merge(EV1, EV2, on = ['event'])
Event_Value

,event,RV14,RV15,RV16,RV17
0,Batter Interference,-0.319625,-0.363838,-0.284649,-0.430019
1,Bunt Groundout,-0.194784,-0.200346,-0.218826,-0.209411
2,Bunt Lineout,-0.303810,-0.421575,-0.352295,-0.328292
3,Bunt Pop Out,-0.316440,-0.354384,-0.342802,-0.373225
4,Catcher Interference,0.380337,0.318276,0.301623,0.399070
5,Double,0.737518,0.752039,0.743467,0.779338
6,Double Play,-0.828774,-0.854665,-0.864981,-0.897164
7,Fan interference,0.633560,0.577453,0.533316,0.590743
8,Field Error,0.462976,0.485460,0.469989,0.493206
9,Fielders Choice,0.698076,0.719351,0.701447,0.764112


In [ ]:
# Compute the correlation matrix for event level run values across all seasons
Event_Value.corr(numeric_only=True)

,RV14,RV15,RV16,RV17
RV14,1.000000,0.998819,0.997951,0.997153
RV15,0.998819,1.000000,0.997570,0.996412
RV16,0.997951,0.997570,1.000000,0.994964
RV17,0.997153,0.996412,0.994964,1.000000


In [ ]:
# For each event, calculate the sum of squares between run values using the data from all four seasons
# and create a column for this sum of squares variable
Event_Value['RV_Bar'] = Event_Value[['RV14', 'RV15', 'RV16', 'RV17']].mean(axis=1)
Event_Value['RV_Sum_of_Squares'] = (Event_Value['RV14']- Event_Value['RV_Bar'])**2 + (Event_Value['RV15']- Event_Value['RV_Bar'])**2\
                                    + (Event_Value['RV16']- Event_Value['RV_Bar'])**2 + (Event_Value['RV17']- Event_Value['RV_Bar'])**2
Event_Value

,event,RV14,RV15,RV16,RV17,RV_Bar,RV_Sum_of_Squares
0,Batter Interference,-0.319625,-0.363838,-0.284649,-0.430019,-0.349533,0.011787
1,Bunt Groundout,-0.194784,-0.200346,-0.218826,-0.209411,-0.205842,0.000334
2,Bunt Lineout,-0.303810,-0.421575,-0.352295,-0.328292,-0.351493,0.007724
3,Bunt Pop Out,-0.316440,-0.354384,-0.342802,-0.373225,-0.346713,0.001694
4,Catcher Interference,0.380337,0.318276,0.301623,0.399070,0.349826,0.006675
5,Double,0.737518,0.752039,0.743467,0.779338,0.753091,0.001025
6,Double Play,-0.828774,-0.854665,-0.864981,-0.897164,-0.861396,0.002402
7,Fan interference,0.633560,0.577453,0.533316,0.590743,0.583768,0.005113
8,Field Error,0.462976,0.485460,0.469989,0.493206,0.477908,0.000577
9,Fielders Choice,0.698076,0.719351,0.701447,0.764112,0.720747,0.002769


# II. Comparing Player Run Values

In [ ]:
# Compute the aggregate player level run values for each season
Player_Value14 = MLBAM14.groupby(['batterId','batterName'])['Run_Value'].sum().reset_index().rename(columns = {"Run_Value": 'RV14'})
Player_Value15 = MLBAM15.groupby(['batterId','batterName'])['Run_Value'].sum().reset_index().rename(columns = {"Run_Value": 'RV15'})
Player_Value16 = MLBAM16.groupby(['batterId','batterName'])['Run_Value'].sum().reset_index().rename(columns = {"Run_Value": 'RV16'})
Player_Value17 = MLBAM17.groupby(['batterId','batterName'])['Run_Value'].sum().reset_index().rename(columns = {"Run_Value": 'RV17'})

In [ ]:
# Merge player run values for each season into one data frame so that only players with run values for all four seasons are included in the data frame.
# The data frame should include each player’s name and then four columns with the run values (one for each season).
P1 = pd.merge(Player_Value14,Player_Value15, on = ['batterName','batterId'])
P2 = pd.merge(Player_Value16,Player_Value17, on = ['batterName','batterId'])
Players = pd.merge(P1,P2, on = ['batterName','batterId'])
Players

,batterId,batterName,RV14,RV15,RV16,RV17
0,112526,Colon,-0.709186,-8.838568,-15.380841,-6.053060
1,134181,Beltre,29.004125,11.918638,29.745115,27.205050
2,136860,Beltran,-6.208773,2.355753,21.279393,-16.883042
3,150029,Werth,37.882267,-3.377653,6.368448,-7.246145
4,282332,Sabathia,-1.118087,-0.472414,-0.944580,-0.292495
...,...,...,...,...,...,...
367,608379,Wacha,-5.461352,-6.021507,-10.598557,-12.735364
368,621035,"Taylor, C",-1.686367,-8.587988,-2.211697,20.588698
369,622072,"Wood, A",-5.490464,-10.162674,-2.623027,-11.017261
370,624577,Puig,31.184967,1.223632,1.871468,5.385486


In [ ]:
# Compute the correlation matrix for event level run values across all seasons
target_columns = ['RV14', 'RV15', 'RV16', 'RV17']
Players[target_columns].corr()

,RV14,RV15,RV16,RV17
RV14,1.000000,0.466299,0.426629,0.322764
RV15,0.466299,1.000000,0.546136,0.510132
RV16,0.426629,0.546136,1.000000,0.457391
RV17,0.322764,0.510132,0.457391,1.000000


In [ ]:
# Run a regression model by regressing player run values from 2017 (dependent variable)
# on player run values from 2014, 2015, and 2016 (independent variables)
import statsmodels.formula.api as smf
lm_1 = smf.ols(formula = 'RV17 ~ RV14 + RV15 + RV16', data=Players).fit()
print(lm_1.summary())

                            OLS Regression Results                            
Dep. Variable:                   RV17   R-squared:                       0.308
Model:                            OLS   Adj. R-squared:                  0.302
Method:                 Least Squares   F-statistic:                     54.61
Date:                Tue, 18 Aug 2026   Prob (F-statistic):           3.20e-29
Time:                        00:23:18   Log-Likelihood:                -1458.4
No. Observations:                 372   AIC:                             2925.
Df Residuals:                     368   BIC:                             2941.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.0472      0.650      0.073      0.9

# III. Comparing Team Run Values

In [ ]:
# For each season’s run expectancy data frame, create a variable “team” to denote the batting team
MLBAM14['team']= np.where(MLBAM14['half']=='top',MLBAM14['away_team'],MLBAM14['home_team'])
MLBAM15['team']= np.where(MLBAM15['half']=='top',MLBAM15['away_team'],MLBAM15['home_team'])
MLBAM16['team']= np.where(MLBAM16['half']=='top',MLBAM16['away_team'],MLBAM16['home_team'])
MLBAM17['team']= np.where(MLBAM17['half']=='top',MLBAM17['away_team'],MLBAM17['home_team'])

In [ ]:
# Compute the aggregate team level run values for each season
Team_Value14= MLBAM14.groupby(['team'])['Run_Value'].sum().reset_index().rename(columns= {"Run_Value": 'RV14'})
Team_Value15= MLBAM15.groupby(['team'])['Run_Value'].sum().reset_index().rename(columns= {"Run_Value": 'RV15'})
Team_Value16= MLBAM16.groupby(['team'])['Run_Value'].sum().reset_index().rename(columns= {"Run_Value": 'RV16'})
Team_Value17= MLBAM17.groupby(['team'])['Run_Value'].sum().reset_index().rename(columns= {"Run_Value": 'RV17'})

In [ ]:
# Merge team run values for each season into one data frame.
# The data frame should include each team’s name and then four columns with the run values (one for each season).
TV1 = pd.merge(Team_Value14,Team_Value15, on = ['team'])
TV2 = pd.merge(Team_Value16,Team_Value17, on = ['team'])
Team_Value = pd.merge(TV1,TV2, on = ['team'])
Team_Value

,team,RV14,RV15,RV16,RV17
0,ana,104.641532,-25.056067,-3.184209,-47.579782
1,ari,-48.253012,31.924551,15.870968,62.616716
2,atl,-81.885752,-108.464134,-84.633901,-28.259905
3,bal,46.586965,29.882644,29.297940,-23.678031
4,bos,-37.383914,65.372953,160.812546,9.445347
5,cha,-1.307578,-79.136738,-41.643638,-47.416033
6,chn,-54.987532,-3.627047,85.327151,71.002092
7,cin,-57.227567,-61.401274,-17.643638,5.420218
8,cle,-3.725729,-4.178644,48.815791,73.780465
9,col,92.456063,47.127799,122.825528,78.616716


In [ ]:
# Compute the correlation matrix for event level run values across all seasons
Team_Value.corr(numeric_only=True)

,RV14,RV15,RV16,RV17
RV14,1.000000,0.363681,0.261814,0.065225
RV15,0.363681,1.000000,0.437610,0.193061
RV16,0.261814,0.437610,1.000000,0.351708
RV17,0.065225,0.193061,0.351708,1.000000


In [ ]:
# Run a regression model by regressing team run values from 2017 (dependent variable)
# on team run values from 2014, 2015, and 2016 (independent variables)
lm_2 = smf.ols(formula = 'RV17 ~ RV14 + RV15 + RV16', data=Team_Value).fit()
print(lm_2.summary())

                            OLS Regression Results                            
Dep. Variable:                   RV17   R-squared:                       0.127
Model:                            OLS   Adj. R-squared:                  0.027
Method:                 Least Squares   F-statistic:                     1.265
Date:                Tue, 18 Aug 2026   Prob (F-statistic):              0.307
Time:                        00:39:04   Log-Likelihood:                -166.93
No. Observations:                  30   AIC:                             341.9
Df Residuals:                      26   BIC:                             347.5
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.4437     12.406     -0.036      0.9